# 7.6 — TF-IDF Text Pipeline

**Chapter 7, sections 7.5.1 and 7.5.2.** The starting point for **Exercise 1** and **Exercise 6**.

**The question this notebook answers:** where does a dimension of a document vector get its
meaning, and what keeps that meaning the same for training and test data?

Four stages — `Tokenizer`, `StopWordsRemover`, `CountVectorizer`, `IDF` — turn a column of film
reviews into a column of sparse TF–IDF vectors. Two of the four are estimators, and what they
learn *is* the feature space: the vocabulary, in order, and one inverse document frequency per
term. The notebook fits that pipeline once, applies it to two datasets, reads the learned
vocabulary back out, and then extends the pipeline with feature selection so that the surviving
dimensions can be printed as words.

Source: `Notebooks old/Spark-Example-23-Mllib-Sentiment Model.ipynb`, cells 16–24 and 36–38,
rebuilt. The original's four classifiers are not part of this chapter and are not carried over;
what is carried over is its feature-engineering pipeline, which is the only code in the
repository that builds one.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-7.6")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 1. The corpus

Fifty thousand film reviews, each labelled positive or negative. A quarter of them are used, so
that the whole notebook runs in well under a minute on a laptop.

The quarter is chosen by **hashing the review text into one hundred buckets** rather than by
`DataFrame.sample`, and the training and test split is made from the same buckets. The reason is
reproducibility. A Spark sample is a per-partition random filter whose result depends on how the
input file happened to be split into partitions, so it can return a different number of rows on
a machine with a different core count — and a `randomSplit` layered on top of an unmaterialized
sample can lose rows outright, because the two sides of the split evaluate the sample twice. A
hash of the content depends on the content and nothing else. Every count in this notebook is
therefore the same on every machine.

In [2]:
import pyspark.sql.functions as F

reviews = spark.read.csv(os.path.join(DATA, "IMDB Dataset.csv.bz2"), header=True)
print("reviews in the file:", reviews.count())
print("columns:", reviews.columns)

# bucket is 0..99, decided by the review text alone.
reviews = (reviews
           .withColumn("bucket", F.crc32(F.col("text")) % 100)
           .filter("bucket < 25")                       # a quarter of the corpus
           .withColumn("label", F.when(F.col("sentiment") == "positive", 1).otherwise(0)))

print("in the sample:", reviews.count())
reviews.select("sentiment", "label", F.substring("text", 1, 66).alias("text")).show(5,
                                                                                    truncate=False)

reviews in the file: 50000
columns: ['text', 'sentiment']


in the sample: 12520
+---------+-----+------------------------------------------------------------------+
|sentiment|label|text                                                              |
+---------+-----+------------------------------------------------------------------+
|negative |0    |Encouraged by the positive comments about this film on here I was |
|negative |0    |Phil the Alien is one of those quirky films where the humour is ba|
|positive |1    |This a fantastic movie of three prisoners who become famous. One o|
|negative |0    |Kind of drawn in by the erotic scenes, only to realize this was on|
|negative |0    |This movie made it into one of my top 10 most awful movies. Horrib|
+---------+-----+------------------------------------------------------------------+
only showing top 5 rows


---

## 2. Cleaning, which is not machine learning

Four regular expressions and a lowercase. HTML tags, anything that is not a letter or a space,
and every word of one or two characters. These are ordinary DataFrame transformations; nothing
here is fitted and nothing here is learned.

In [3]:
cleaned = F.regexp_replace(F.col("text"), r'<[^>]+>', "")      # HTML tags
cleaned = F.regexp_replace(cleaned, r"[^a-zA-Z ]", "")         # digits and punctuation
cleaned = F.regexp_replace(cleaned, r"\b\w{1,2}\b", "")        # one- and two-letter words
reviews = reviews.withColumn("text_c", F.lower(cleaned)).cache()

reviews.select(F.substring("text_c", 1, 90).alias("text_c")).show(3, truncate=False)

+------------------------------------------------------------------------------------------+
|text_c                                                                                    |
+------------------------------------------------------------------------------------------+
|encouraged  the positive comments about this film  here  was looking forward  watching thi|
|phil the alien  one  those quirky films where the humour  based around the oddness  everyt|
|this  fantastic movie  three prisoners who become famous one  the actors  george clooney a|
+------------------------------------------------------------------------------------------+
only showing top 3 rows


Look at the spacing in that output. Every removed word left the spaces that surrounded it, so the
text now contains runs of consecutive whitespace. Section 5 comes back to what that does.

In [4]:
# The split is a filter on the same bucket column: buckets 5-24 train, buckets 0-4 test.
train_docs = reviews.filter("bucket >= 5").cache()
test_docs = reviews.filter("bucket < 5")

n_train, n_test = train_docs.count(), test_docs.count()
print("training documents:", n_train, " test documents:", n_test)

assert n_train + n_test == reviews.count()
print("the two sides account for every sampled review")

training documents: 10018  test documents: 2502
the two sides account for every sampled review


---

## 3. The four-stage pipeline

* `Tokenizer` splits on whitespace and lowercases. A **transformer**.
* `StopWordsRemover` drops the words on a list. A **transformer**; the list is a parameter, not
  something learned.
* `CountVectorizer` is an **estimator**. Its `fit` scans the corpus, counts every term, and keeps
  the `vocabSize` commonest ones as an ordered list. That list is the feature space.
* `IDF` is an **estimator**. Its `fit` counts how many documents contain each term.

Two estimators, so `fit` makes at least two passes over the training documents — which is why
they were cached above.

In [5]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF

tokenizer = Tokenizer(inputCol="text_c", outputCol="words")
remover = StopWordsRemover(inputCol=tokenizer.getOutputCol(), outputCol="filtered")
countVectorizer = CountVectorizer(inputCol=remover.getOutputCol(),
                                  outputCol="rawFeatures", vocabSize=1000)
idf = IDF(inputCol=countVectorizer.getOutputCol(), outputCol="featuresIDF")

pipeline_p = Pipeline(stages=[tokenizer, remover, countVectorizer, idf])
data_model = pipeline_p.fit(train_docs)

In [6]:
transformed_data = data_model.transform(train_docs)
transformed_test = data_model.transform(test_docs)

transformed_data.select("label", "featuresIDF").show(3, truncate=70)

+-----+----------------------------------------------------------------------+
|label|                                                           featuresIDF|
+-----+----------------------------------------------------------------------+
|    0|(1000,[0,2,3,9,17,23,24,28,33,35,47,51,53,68,84,91,105,113,117,118,...|
|    0|(1000,[0,1,2,3,19,24,25,31,42,46,57,60,66,67,76,78,80,101,138,158,2...|
|    0|(1000,[0,2,3,4,12,17,25,28,41,50,80,83,87,91,129,166,180,203,260,29...|
+-----+----------------------------------------------------------------------+
only showing top 3 rows


Those two lines are the argument. **One** fitted object, applied to two datasets. The vocabulary
was learned from the training corpus alone and is applied unchanged to the test corpus, so
dimension 17 means the same term on both sides.

Had a second `CountVectorizer` been fitted on the test set, it would have learned a different
vocabulary from a different distribution of words, and dimension 17 would have meant something
else. The model would then be scored on vectors that do not correspond to the ones it was
trained on — and no error would be raised, because nothing about the shapes would be wrong.

---

## 4. Reading the fitted stages

The pipeline model is inspectable. Each fitted stage carries the state it learned, and printing
the stage list shows that state rather than the code that produced it.

In [7]:
data_model.stages

[Tokenizer_54236b583bb0,
 StopWordsRemover_b7a2491787e6,
 CountVectorizerModel: uid=CountVectorizer_cd65391d09f9, vocabularySize=1000,
 IDFModel: uid=IDF_93adff7336a4, numDocs=10018, numFeatures=1000]

In [8]:
data_model.stages[2].vocabulary[:20]

['',
 'movie',
 'film',
 'one',
 'like',
 'good',
 'even',
 'time',
 'really',
 'see',
 'story',
 'much',
 'get',
 'well',
 'people',
 'great',
 'also',
 'bad',
 'dont',
 'first']

Three things in that pair of outputs.

**The estimators announce what they learned.** `CountVectorizerModel` reports its vocabulary
size and `IDFModel` reports the number of documents it counted and the number of features it
will emit. Neither number appears anywhere in the code above; both are properties of the fitted
object.

**The vocabulary is ordered by corpus frequency**, and the order is the feature space. Entry *i*
of the vocabulary is dimension *i* of every document vector, which is what makes section 6
possible.

**The first entry is the empty string.** That is not a Spark quirk. It comes from the cleaning in
section 2, several stages earlier, and section 5 traces it.

In [9]:
# The exact correspondence, checked on one document rather than asserted.
vocabulary = data_model.stages[2].vocabulary
row = transformed_data.select("rawFeatures").first()["rawFeatures"]

print(f"{'dim':>5}  {'term':<12} {'count in this document':>22}")
for i in sorted(int(j) for j in row.indices)[:8]:
    print(f"{i:>5}  {vocabulary[i]!r:<12} {int(row[i]):>22}")

  dim  term         count in this document
    0  ''                               34
    2  'film'                            4
    3  'one'                             2
    9  'see'                             1
   17  'bad'                             1
   23  'way'                             1
   24  'films'                           2
   28  'seen'                            1


In [10]:
# The IDF weighting, checked against the definition the chapter gives:
#   IDF(t, D) = log( (|D| + 1) / (DF(t, D) + 1) )
import math

idf_model = data_model.stages[3]
num_docs = idf_model.numDocs
term = "movie"
t = vocabulary.index(term)

doc_freq = (transformed_data
            .filter(F.array_contains("filtered", term))
            .count())

expected = math.log((num_docs + 1) / (doc_freq + 1))
print(f"term {term!r} is dimension {t}")
print(f"documents containing it: {doc_freq} of {num_docs}")
print(f"IDF from the formula:    {expected:.6f}")
print(f"IDF from the model:      {idf_model.idf[t]:.6f}")
assert abs(expected - idf_model.idf[t]) < 1e-9

term 'movie' is dimension 1
documents containing it: 5980 of 10018
IDF from the formula:    0.515896
IDF from the model:      0.515896


---

## 5. Where the empty string came from

The cleaning of section 2 replaced each removed word with the empty string, which left the spaces
that had surrounded it. `Tokenizer` splits on whitespace and emits an empty token for each run of
it. `StopWordsRemover` has no reason to drop an empty token, so it survives to the vectorizer,
where it is the single most frequent "term" in the corpus and therefore takes dimension 0 of
every document vector.

The fault is four stages upstream of where it becomes visible, which is the general shape of the
problem rather than a detail of this corpus.

In [11]:
sample_tokens = (data_model.transform(train_docs.limit(1))
                 .select("words").first()["words"])
print("first 14 tokens:", sample_tokens[:14])
print("empty tokens in this one document:", sample_tokens.count(""))

first 14 tokens: ['encouraged', '', 'the', 'positive', 'comments', 'about', 'this', 'film', '', 'here', '', 'was', 'looking', 'forward']
empty tokens in this one document: 34


### Two fixes, both one line

Either collapse the runs of whitespace before tokenizing, or tokenize on a pattern that cannot
produce an empty token in the first place. Both are shown; both remove the empty string from the
vocabulary and shift every other term up one dimension.

In [12]:
from pyspark.ml.feature import RegexTokenizer

squeezed = train_docs.withColumn("text_t",
                                 F.trim(F.regexp_replace(F.col("text_c"), r"\s+", " ")))

fix_a = Pipeline(stages=[
    Tokenizer(inputCol="text_t", outputCol="words"),
    StopWordsRemover(inputCol="words", outputCol="filtered"),
    CountVectorizer(inputCol="filtered", outputCol="rawFeatures", vocabSize=1000),
]).fit(squeezed)

fix_b = Pipeline(stages=[
    RegexTokenizer(inputCol="text_c", outputCol="words", pattern=r"\W+"),
    StopWordsRemover(inputCol="words", outputCol="filtered"),
    CountVectorizer(inputCol="filtered", outputCol="rawFeatures", vocabSize=1000),
]).fit(train_docs)

print("as cleaned in section 2   ", vocabulary[:8])
print("whitespace squeezed       ", fix_a.stages[2].vocabulary[:8])
print("RegexTokenizer on \\W+     ", fix_b.stages[2].vocabulary[:8])

as cleaned in section 2    ['', 'movie', 'film', 'one', 'like', 'good', 'even', 'time']
whitespace squeezed        ['movie', 'film', 'one', 'like', 'good', 'even', 'time', 'really']
RegexTokenizer on \W+      ['movie', 'film', 'one', 'like', 'good', 'even', 'time', 'really']


In [13]:
assert "" not in fix_a.stages[2].vocabulary
assert "" not in fix_b.stages[2].vocabulary
print("both fixes remove the empty string from the vocabulary")

both fixes remove the empty string from the vocabulary


---

## 6. Overriding parameters at fit time

The same pipeline object, refitted with different settings, without editing a stage. A `ParamMap`
is a set of (parameter, value) pairs, and a parameter belongs to a specific object instance, so
one map can carry settings for several stages at once without ambiguity.

This is the mechanism a parameter grid is built on: a grid is a collection of such maps, and a
search is the same estimator fitted once per map.

In [14]:
paramMap = {
    countVectorizer.getParam("vocabSize"): 300,
    idf.getParam("minDocFreq"): 5,
}

small_model = pipeline_p.fit(train_docs, paramMap)
small_model.stages

[Tokenizer_54236b583bb0,
 StopWordsRemover_b7a2491787e6,
 CountVectorizerModel: uid=CountVectorizer_cd65391d09f9, vocabularySize=300,
 IDFModel: uid=IDF_93adff7336a4, numDocs=10018, numFeatures=300]

In [15]:
# The stage objects were not modified; only this one fit used the overrides.
print("countVectorizer.vocabSize is still", countVectorizer.getVocabSize())
print("the refitted model's vocabulary size is", len(small_model.stages[2].vocabulary))

countVectorizer.vocabSize is still 1000
the refitted model's vocabulary size is 300


---

## 7. A fifth stage, and reading the selected features back as words

`ChiSqSelector` keeps the `numTopFeatures` dimensions most strongly associated with the label by
a chi-squared test. Its output is a shorter vector — and because the fitted `CountVectorizerModel`
is still in the same pipeline, each surviving index can be looked up in the vocabulary.

The cleaning here uses the whitespace fix of section 5, so that dimension 0 is a word rather than
the empty string.

In [16]:
from pyspark.ml.feature import ChiSqSelector

train_fixed = train_docs.withColumn(
    "text_c", F.trim(F.regexp_replace(F.col("text_c"), r"\s+", " ")))

tokenizer5 = Tokenizer(inputCol="text_c", outputCol="words")
remover5 = StopWordsRemover(inputCol="words", outputCol="filtered")
cv5 = CountVectorizer(inputCol="filtered", outputCol="rawFeatures", vocabSize=1000)
idf5 = IDF(inputCol="rawFeatures", outputCol="featuresIDF")
selector = ChiSqSelector(numTopFeatures=200, featuresCol="featuresIDF",
                         outputCol="features", labelCol="label")

selected_model = Pipeline(stages=[tokenizer5, remover5, cv5, idf5, selector]).fit(train_fixed)
selected_model.stages

[Tokenizer_d56aa6686158,
 StopWordsRemover_fa1f2355ef29,
 CountVectorizerModel: uid=CountVectorizer_67ce3b8bda51, vocabularySize=1000,
 IDFModel: uid=IDF_9113d22b606d, numDocs=10018, numFeatures=1000,
 ChiSqSelectorModel: uid=ChiSqSelector_793c60b8403f, numSelectedFeatures=200]

In [17]:
vocabulary5 = selected_model.stages[2].vocabulary
selected_indexes = selected_model.stages[4].selectedFeatures
selected_words = [vocabulary5[i] for i in selected_indexes]

print(f"{len(vocabulary5)} terms reduced to {len(selected_words)}")
print()
print("the first thirty:")
print(selected_words[:30])

1000 terms reduced to 200

the first thirty:
['movie', 'like', 'even', 'really', 'well', 'great', 'also', 'bad', 'dont', 'movies', 'make', 'love', 'best', 'acting', 'plot', 'life', 'better', 'still', 'say', 'something', 'watching', 'doesnt', 'thing', 'years', 'didnt', 'new', 'nothing', 'actually', 'want', 'cant']


In [18]:
# The words a reader would expect a sentiment classifier to keep. None of them is in the
# first thirty, because those are the frequent terms; these are the discriminating ones.
opinion = ["excellent", "boring", "awful", "brilliant", "waste", "masterpiece", "pathetic"]
for w in opinion:
    where = selected_words.index(w) if w in selected_words else None
    print(f"  {w:<12} {'kept, rank ' + str(where) if where is not None else 'dropped'}")

  excellent    kept, rank 68
  boring       kept, rank 80
  awful        kept, rank 88
  brilliant    kept, rank 110
  waste        kept, rank 102
  masterpiece  kept, rank 178
  pathetic     dropped


In [19]:
# The dimensions really did shrink, and the vector really is sparse.
out = selected_model.transform(train_fixed).select("featuresIDF", "features").first()
print("before selection:", out["featuresIDF"].size, "dimensions,",
      len(out["featuresIDF"].indices), "non-zero")
print("after selection: ", out["features"].size, "dimensions,",
      len(out["features"].indices), "non-zero")

before selection: 1000 dimensions, 46 non-zero
after selection:  200 dimensions, 15 non-zero


This is the clearest demonstration in the chapter that a feature vector's dimensions mean
something. A model fitted on these two hundred features can be inspected term by term, because
the vocabulary travelled inside the same fitted object as the selector that indexed into it.
Separate the two and the indices become anonymous.

---

## 8. What a term seen only in the test set does

Nothing. It is not in the vocabulary, so no dimension corresponds to it and it is silently
ignored. This is the desired behaviour: the alternative, a new dimension the model has never
been trained on, is worse.

In [20]:
invented = spark.createDataFrame(
    [("this film was utterly zzyzxlike and quite wonderful", 1)], ["text_c", "label"])

vec = data_model.transform(invented).select("rawFeatures").first()["rawFeatures"]
present = [vocabulary[int(i)] for i in sorted(vec.indices)]

print("terms in the sentence that made it into a dimension:", present)
print("'zzyzxlike' in the vocabulary:", "zzyzxlike" in vocabulary)
print("vector length is still:", vec.size)

terms in the sentence that made it into a dimension: ['film', 'quite', 'wonderful', 'utterly']
'zzyzxlike' in the vocabulary: False
vector length is still: 1000


---

## What to take away

Two of the four stages are estimators, and between them they define the feature space: an ordered
vocabulary and one inverse document frequency per term. Both are learned from the training corpus
once and reused unchanged everywhere else, which is what `PipelineModel` is for.

The numbers from this run, on the hashed quarter of the corpus: **12,520 reviews**, split into
**10,018 training** and **2,502 test** documents, a vocabulary of **1,000** terms, and `IDFModel`
reporting `numDocs=10018, numFeatures=1000`. They are reproducible on any machine, which the
`sample`-and-`randomSplit` version they replace was not. The empty string is the most
frequent term and takes dimension 0, and it does so because of a regular expression four stages
earlier.

In [21]:
spark.stop()